In [1]:
#Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


In [2]:
#Leitura dos arquivos CSV
df_prod = pd.read_csv(r"C:\Users\Bruno\Documents\Projeto-Em-Business-Intelligence-e-Analytics\data\dados_producao_agricola\dados_producao_noroeste_RS.csv")
df_clima = pd.read_csv(r"C:\Users\Bruno\Documents\Projeto-Em-Business-Intelligence-e-Analytics\data\dados_metereologicos\dados_rs_tratados\dados_meteorologia_RS.csv")
df_geo = pd.read_csv(r"C:\Users\Bruno\Documents\Projeto-Em-Business-Intelligence-e-Analytics\data\daos_geograficos\dados_tratados\dados_geograficos_RS.csv")



In [3]:
#Padronização dos dados
df_geo['codigo_municipio'] = df_geo['codigo_municipio'].astype(str)
df_prod['cod_municipio'] = df_prod['cod_municipio'].astype(str)

df_geo['nome_municipio'] = df_geo['nome_municipio'].str.upper().str.strip()
df_clima['cidade'] = df_clima['cidade'].str.upper().str.strip()

df_geo['nome_municipio'] = df_geo['nome_municipio'].str.upper().str.strip()


In [4]:
#Tratamento dos dados de produção agricola
df_prod = df_prod[df_prod['variavel'] == 'Área plantada']

df_prod = df_prod.pivot_table(
    index=['cod_municipio', 'ano'],
    columns='produto',
    values='valor',
    aggfunc='sum'
).reset_index()

df_prod.columns.name = None

display(df_prod)

,cod_municipio,ano,Fumo,Milho,Soja,Trigo
0,4300034,2010,NaN,1500.0,2100.0,700.0
1,4300034,2011,NaN,3000.0,5000.0,700.0
2,4300034,2012,NaN,3000.0,8000.0,600.0
3,4300034,2013,NaN,2000.0,15000.0,200.0
4,4300034,2014,NaN,3000.0,18000.0,400.0
...,...,...,...,...,...,...
7355,4323770,2024,NaN,150.0,130.0,50.0
7356,4323804,2010,NaN,10.0,NaN,NaN
7357,4323804,2011,NaN,10.0,NaN,NaN
7358,4323804,2012,NaN,5.0,NaN,NaN


In [5]:
#Tratamentod de dados dmeteorológicos (diário → anual)
df_clima['data'] = pd.to_datetime(df_clima['data'], errors='coerce')

df_clima_ano = df_clima.groupby(['cidade', 'ano']).agg({
    'precipitacao': ['sum', 'mean', 'std'],
    'temperatura': ['mean', 'max', 'min'],
    'umidade': ['mean'],
    'vento': ['mean']
}).reset_index()

# flatten colunas
df_clima_ano.columns = [
    'cidade', 'ano',
    'prec_total', 'prec_media', 'prec_std',
    'temp_media', 'temp_max', 'temp_min',
    'umidade_media',
    'vento_media'
]

df_clima_ano['cidade'] = df_clima_ano['cidade'].str.upper().str.strip()

display(df_clima_ano)

,cidade,ano,prec_total,prec_media,prec_std,temp_media,temp_max,temp_min,umidade_media,vento_media
0,ALEGRETE,2014,349.0,0.956164,4.808217,19.707557,35.0,4.50,75.707340,2.206605
1,ALEGRETE,2015,334.0,0.915068,3.577389,19.322688,33.0,5.00,75.039734,2.199992
2,ALEGRETE,2016,337.0,0.920765,3.819454,19.043646,34.2,4.00,73.903119,2.142441
3,ALEGRETE,2017,201.0,0.550685,2.362896,19.907068,35.0,5.50,74.060348,2.244415
4,ALEGRETE,2018,317.0,0.868493,4.017027,19.164329,33.0,1.00,75.248700,2.175064
...,...,...,...,...,...,...,...,...,...,...
464,VACARIA,2020,284.0,0.775956,3.360377,16.294265,29.0,2.75,75.245073,3.011367
465,VACARIA,2021,364.0,0.997260,3.330263,15.544193,30.0,0.00,80.771174,2.744458
466,VACARIA,2022,364.0,0.997260,3.414174,15.065039,29.0,-1.00,81.502311,3.241056
467,VACARIA,2023,370.0,1.013699,3.910022,15.763426,29.0,1.00,82.406444,3.093757


In [6]:

df_clima_geo = df_clima_ano.merge(
    df_geo,
    left_on='cidade',
    right_on='nome_municipio',
    how='left'
)

df_clima_regiao = df_clima_geo.groupby(
    ['regiao_imediata', 'ano']
).agg({
    'prec_total': 'mean',
    'temp_media': 'mean',
    'umidade_media': 'mean',
    'vento_media': 'mean'
}).reset_index()

df_clima_regiao['regiao_imediata'] = df_clima_regiao['regiao_imediata'].astype(int)

display(df_clima_regiao)

,regiao_imediata,ano,prec_total,temp_media,umidade_media,vento_media
0,430001,2014,280.5,20.458153,75.582822,3.192411
1,430001,2015,357.5,20.297816,79.646504,3.275483
2,430001,2016,264.0,18.793295,79.890514,3.263076
3,430001,2017,300.0,20.460919,78.956537,3.435944
4,430001,2018,265.5,19.845050,79.766929,3.306482
...,...,...,...,...,...,...
203,430040,2020,196.0,18.985047,71.913338,2.700686
204,430040,2021,186.0,18.807567,73.043650,2.642291
205,430040,2022,219.5,18.224172,75.827485,2.545405
206,430040,2023,446.5,19.128378,77.198409,2.446412


In [7]:
#Merge entre as bases de dados

df = df_prod.merge(
    df_geo,
    left_on='cod_municipio',
    right_on='codigo_municipio',
    how='left'
)

df = df.merge(
    df_clima_regiao,
    on=['regiao_imediata', 'ano'],
    how='left'
)

display(df)

,cod_municipio,ano,Fumo,Milho,Soja,Trigo,uf,nome_uf,regiao_intermediaria,nome_regiao_intermediaria,regiao_imediata,nome_regiao_imediata,codigo_municipio,nome_municipio,uf_sigla,prec_total,temp_media,umidade_media,vento_media
0,4300034,2010,NaN,1500.0,2100.0,700.0,43,Rio Grande do Sul,4302,Pelotas,430010,Bagé,4300034,ACEGUÁ,RS,NaN,NaN,NaN,NaN
1,4300034,2011,NaN,3000.0,5000.0,700.0,43,Rio Grande do Sul,4302,Pelotas,430010,Bagé,4300034,ACEGUÁ,RS,NaN,NaN,NaN,NaN
2,4300034,2012,NaN,3000.0,8000.0,600.0,43,Rio Grande do Sul,4302,Pelotas,430010,Bagé,4300034,ACEGUÁ,RS,NaN,NaN,NaN,NaN
3,4300034,2013,NaN,2000.0,15000.0,200.0,43,Rio Grande do Sul,4302,Pelotas,430010,Bagé,4300034,ACEGUÁ,RS,NaN,NaN,NaN,NaN
4,4300034,2014,NaN,3000.0,18000.0,400.0,43,Rio Grande do Sul,4302,Pelotas,430010,Bagé,4300034,ACEGUÁ,RS,246.0,18.736643,76.981346,3.356273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7355,4323770,2024,NaN,150.0,130.0,50.0,43,Rio Grande do Sul,4308,Santa Cruz do Sul - Lajeado,430041,Lajeado,4323770,WESTFÁLIA,RS,NaN,NaN,NaN,NaN
7356,4323804,2010,NaN,10.0,NaN,NaN,43,Rio Grande do Sul,4301,Porto Alegre,430003,Tramandaí - Osório,4323804,XANGRI-LÁ,RS,NaN,NaN,NaN,NaN
7357,4323804,2011,NaN,10.0,NaN,NaN,43,Rio Grande do Sul,4301,Porto Alegre,430003,Tramandaí - Osório,4323804,XANGRI-LÁ,RS,NaN,NaN,NaN,NaN
7358,4323804,2012,NaN,5.0,NaN,NaN,43,Rio Grande do Sul,4301,Porto Alegre,430003,Tramandaí - Osório,4323804,XANGRI-LÁ,RS,NaN,NaN,NaN,NaN


In [8]:
#Feature Engineering
df['area_total'] = df[['Milho', 'Soja', 'Trigo']].sum(axis=1)
df['area_total'] = df['area_total'].replace(0, np.nan)

# proporções
df['perc_soja'] = df['Soja'] / df['area_total']
df['perc_milho'] = df['Milho'] / df['area_total']
df['perc_trigo'] = df['Trigo'] / df['area_total']

# clima derivado (ajustado)
df['precipitacao_por_dia'] = df['prec_total'] / 365

In [9]:
#Limpeza final
df_modelo = df.dropna()

display(df_modelo)

,cod_municipio,ano,Fumo,Milho,Soja,Trigo,uf,nome_uf,regiao_intermediaria,nome_regiao_intermediaria,...,uf_sigla,prec_total,temp_media,umidade_media,vento_media,area_total,perc_soja,perc_milho,perc_trigo,precipitacao_por_dia
34,4300109,2014,5100.0,3000.0,600.0,6.0,43,Rio Grande do Sul,4303,Santa Maria,...,RS,451.0,19.675090,82.101726,2.068589,3606.0,0.166389,0.831947,0.001664,1.235616
35,4300109,2015,5000.0,3500.0,700.0,10.0,43,Rio Grande do Sul,4303,Santa Maria,...,RS,409.0,19.590451,83.557197,2.100970,4210.0,0.166271,0.831354,0.002375,1.120548
36,4300109,2016,4500.0,3450.0,850.0,8.0,43,Rio Grande do Sul,4303,Santa Maria,...,RS,347.0,18.264684,82.450492,1.962345,4308.0,0.197307,0.800836,0.001857,0.950685
37,4300109,2017,4600.0,3500.0,900.0,2.0,43,Rio Grande do Sul,4303,Santa Maria,...,RS,436.0,19.903276,79.466251,2.225965,4402.0,0.204453,0.795093,0.000454,1.194521
41,4300109,2021,4500.0,3500.0,1000.0,100.0,43,Rio Grande do Sul,4303,Santa Maria,...,RS,270.0,18.746458,78.339512,2.153844,4600.0,0.217391,0.760870,0.021739,0.739726
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7291,4323507,2020,200.0,900.0,1500.0,350.0,43,Rio Grande do Sul,4306,Passo Fundo,...,RS,247.0,19.697355,69.958561,1.085507,2750.0,0.545455,0.327273,0.127273,0.676712
7292,4323507,2021,200.0,1000.0,1500.0,450.0,43,Rio Grande do Sul,4306,Passo Fundo,...,RS,228.0,19.915807,72.800689,0.739205,2950.0,0.508475,0.338983,0.152542,0.624658
7293,4323507,2022,150.0,1000.0,1550.0,500.0,43,Rio Grande do Sul,4306,Passo Fundo,...,RS,249.0,18.902324,74.348678,0.415861,3050.0,0.508197,0.327869,0.163934,0.682192
7294,4323507,2023,100.0,900.0,1600.0,500.0,43,Rio Grande do Sul,4306,Passo Fundo,...,RS,511.0,20.276331,78.039879,1.147981,3000.0,0.533333,0.300000,0.166667,1.400000


In [10]:
y = df_modelo['Soja'] 

In [11]:
features = [
    'prec_total',
    'temp_media',
    'umidade_media',
    'vento_media',
    'precipitacao_por_dia',
    'Milho',
    'Trigo',
    'perc_milho',
    'perc_trigo'
]

X = df_modelo[features]

# ================================
# ✂️ TRAIN / TEST
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
# 🤖 MODELO
# ================================
modelo = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

modelo.fit(X_train, y_train)

# ================================
# 🔮 PREDIÇÃO
# ================================
y_pred = modelo.predict(X_test)

# ================================
# 📈 MÉTRICAS
# ================================
print("R2:", r2_score(y_test, y_pred))
rmse = mean_squared_error(y_test, y_pred) ** 0.5
print("RMSE:", rmse)

# ================================
# 📊 IMPORTÂNCIA DAS VARIÁVEIS
# ================================
importancias = pd.DataFrame({
    'variavel': X.columns,
    'importancia': modelo.feature_importances_
}).sort_values(by='importancia', ascending=False)

print(importancias)



R2: 0.9433154323182887
RMSE: 6016.396215600129
               variavel  importancia
7            perc_milho     0.472286
5                 Milho     0.241645
6                 Trigo     0.187620
8            perc_trigo     0.056996
3           vento_media     0.027537
1            temp_media     0.006833
2         umidade_media     0.004666
0            prec_total     0.001219
4  precipitacao_por_dia     0.001199


In [ ]:
# ================================
# 💾 SALVAR RESULTADO
# ================================
df_modelo = df_modelo.copy()
df_modelo['pred_soja'] = modelo.predict(X)

df_modelo.to_csv(
    "../data/dados_tratados/base_modelo_final.csv",
    index=False
)

In [ ]:

# ================================
# 🧹 PADRONIZAÇÃO
# ================================


# ================================
# 📊 TRATAR CLIMA (DIÁRIO → ANUAL)
# ================================

# ================================
# 📊 TRATAR PRODUÇÃO
# ================================


# ================================
# 🔗 MERGE GEO
# ================================

# ================================
# 🔗 MERGE CLIMA
# ================================


# ================================
# 🧠 FEATURE ENGINEERING
# ================================
# total culturas

# ================================
# 🧹 LIMPEZA FINAL
# ================================
df_modelo = df.dropna()

# ================================
# 🎯 VARIÁVEL ALVO
# ================================
y = df_modelo['Soja']  # pode trocar por area_total

# ================================
# 📊 FEATURES
# ================================
features = [
    'prec_total',
    'prec_media',
    'prec_std',
    'temp_media',
    'temp_max',
    'temp_min',
    'amplitude_termica',
    'umidade_media',
    'vento_media',
    'precipitacao_por_dia',
    'Milho',
    'Trigo',
    'perc_milho',
    'perc_trigo'
]

X = df_modelo[features]

# ================================
# ✂️ TRAIN / TEST
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
# 🤖 MODELO
# ================================
modelo = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

modelo.fit(X_train, y_train)

# ================================
# 🔮 PREDIÇÃO
# ================================
y_pred = modelo.predict(X_test)

# ================================
# 📈 MÉTRICAS
# ================================
print("R2:", r2_score(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))

# ================================
# 📊 IMPORTÂNCIA DAS VARIÁVEIS
# ================================
importancias = pd.DataFrame({
    'variavel': X.columns,
    'importancia': modelo.feature_importances_
}).sort_values(by='importancia', ascending=False)

print(importancias)

# ================================
# 💾 SALVAR RESULTADO
# ================================
df_modelo['pred_soja'] = modelo.predict(X)
df_modelo.to_csv("../data/dados_tratados/base_modelo_final.csv", index=False)